In [2]:
import pandas as pd
import os

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import numpy as np
import joblib


df = pd.read_csv("../outputs/healthcare_cost_dataset.csv")

print("Dataset Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

features = [
    "Age",
    "Gender",
    "Region",
    "Socioeconomic_Status",
    "Primary_Diagnosis",
    "Blood_Glucose_mg_dL",
    "HbA1c_%",
    "Total_Cholesterol_mg_dL",
    "Treatment_Type",
    "Treatment_Outcome",
    "Imaging_Type",
    "Hospital_Type",
    "Insurance_Covered",
    "BMI"
]

target = "Treatment_Cost_INR"


X = df[features]
y = df[target]


print("\nFeatures:")
print(X.head())

print("\nTarget:")
print(y.head())


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("\nTraining Data:", X_train.shape)
print("Testing Data:", X_test.shape)


categorical_features = [
    "Gender",
    "Region",
    "Socioeconomic_Status",
    "Primary_Diagnosis",
    "Treatment_Type",
    "Treatment_Outcome",
    "Imaging_Type",
    "Hospital_Type",
    "Insurance_Covered"
]

numerical_features = [
    "Age",
    "Blood_Glucose_mg_dL",
    "HbA1c_%",
    "Total_Cholesterol_mg_dL",
    "BMI"
]


numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)


models = {

    "Linear Regression": LinearRegression(),

    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )
}


results = {}

print("\n========== MODEL TRAINING ==========")

for name, model in models.items():

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ]
    )

    print("\nTraining:", name)

    pipeline.fit(X_train, y_train)

    predictions = pipeline.predict(X_test)

    mae = mean_absolute_error(
        y_test,
        predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            predictions
        )
    )

    r2 = r2_score(
        y_test,
        predictions
    )

    results[name] = {
        "MAE": mae,
        "RMSE": rmse,
        "R2 Score": r2,
        "Pipeline": pipeline
    }

    print("MAE:", round(mae, 2))
    print("RMSE:", round(rmse, 2))
    print("R2 Score:", round(r2, 4))




print("\n========== MODEL COMPARISON ==========")

for name, result in results.items():

    print(
        name,
        " | MAE:",
        round(result["MAE"], 2),
        " | RMSE:",
        round(result["RMSE"], 2),
        " | R2:",
        round(result["R2 Score"], 4)
    )



best_model_name = max(
    results,
    key=lambda x: results[x]["R2 Score"]
)

best_model = results[best_model_name]["Pipeline"]

print("\n======================================")
print("BEST MODEL:", best_model_name)
print("R2 SCORE:", round(
    results[best_model_name]["R2 Score"],
    4
))
print("MAE:", round(
    results[best_model_name]["MAE"],
    2
))
print("RMSE:", round(
    results[best_model_name]["RMSE"],
    2
))
print("======================================")



os.makedirs("../outputs", exist_ok=True)

joblib.dump(
    best_model,
    "../outputs/best_medical_cost_model.pkl"
)

print("\nBest model saved successfully!")

print(
    "Location: ../outputs/best_medical_cost_model.pkl"
)

Dataset Shape: (100000, 21)

Columns:
['Patient_ID', 'Age', 'Gender', 'Region', 'Socioeconomic_Status', 'Occupation', 'Visit_Date', 'Symptoms', 'Primary_Diagnosis', 'Blood_Glucose_mg_dL', 'HbA1c_%', 'Total_Cholesterol_mg_dL', 'Treatment_Type', 'Treatment_Outcome', 'Imaging_Type', 'Imaging_Findings', 'Hospital_Type', 'Insurance_Covered', 'BMI', 'Treatment_Cost_INR', 'Cost_Category']

Features:
   Age  Gender          Region Socioeconomic_Status       Primary_Diagnosis  \
0   53  Female  Andhra Pradesh                  Low                  Dengue   
1   36    Male      Puducherry                  Low                 Malaria   
2   28    Male     Maharashtra                  Low            Hypertension   
3   34  Female       Karnataka                  Low  Cardiovascular Disease   
4   48  Female      Puducherry                 High            Hypertension   

   Blood_Glucose_mg_dL  HbA1c_%  Total_Cholesterol_mg_dL       Treatment_Type  \
0                100.9      5.9                 